# What this notebook is about

In this notebook we will analyze the traffic generated by a browser for accessing a web page, in particular, for understanding the usage of the DNS to obtain IP addresses of web servers.




# Preparation (install `Scapy` and `tshark`)

## Install and configure the necessary software


In [ ]:
!pip install scapy
from scapy.all import *

In [ ]:
!apt install tshark

## Define Scapy functions (execute only once)

In [ ]:
dns_type_to_name = {
    1: "A",       # IPv4 address
    2: "NS",      # Name Server
    5: "CNAME",   # Canonical Name
    6: "SOA",     # Start of Authority
    12: "PTR",    # Pointer record
    15: "MX",     # Mail exchange
    16: "TXT",    # Text record
    28: "AAAA",   # IPv6 address
    33: "SRV",    # Service locator
    35: "NAPTR",  # Naming authority pointer
    36: "KX",     # Key Exchanger
    43: "DS",     # Delegation Signer
    46: "RRSIG",  # DNSSEC signature
    47: "NSEC",   # Next Secure record
    48: "DNSKEY", # DNS Key record
    257: "CAA"    # Certification Authority Authorization
}

def display_frames(packets, indexes):
  for index in indexes:
    print('-' * 20)
    index = index-1
    if index < 0 or index >= len(packets):
      print(f"Invalid index: {index}")
      continue

    packet = packets[index]

    if packet.haslayer(IP):
      ip_layer = packet.getlayer(IP)
      ip_address_src = ip_layer.src
      ip_address_dst = ip_layer.dst
      print(f"Frame {index+1}: {ip_address_src} ----> {ip_address_dst}")
    else:
      print(f"Frame {index+1} No IP layer found")
      continue

    if packet.haslayer(DNS) and packet.getlayer(DNS).qr == 0:
      # The second check is true only if the DNS message is a request (detail of the DNS protocol)
      dns_layer = packet.getlayer(DNS)
      queried_rr = dns_layer.qd.qname.decode('utf-8')
      query_type = dns_type_to_name [dns_layer.qd.qtype] # Scapy dict for mapping DNS types to their textual representation
      transaction_id = dns_layer.id
      print(f"DNS Request: (xid: {transaction_id})\n{queried_rr} {query_type}")
    elif packet.haslayer(DNS) and packet.getlayer(DNS).ancount > 0:
      # The second check is true only if the DNS message is a response (detail of the DNS protocol)
      dns_layer = packet.getlayer(DNS)
      formatted_res = ''
      for i in range(dns_layer.ancount):
        rr = dns_layer.an[i]
        rr_name = rr.rrname.decode('utf-8')
        #rr_type = str(rr.type)
        rr_type = dns_type_to_name[rr.type] # Scapy dict for mapping DNS types to their textual representation
        rr_rdata = str(rr.rdata)
        transaction_id = str(dns_layer.id)
        formatted_res = formatted_res + rr_name + ' ' + rr_type + ' ' + rr_rdata + '\n'
      print(f"DNS Response: (xid: {transaction_id})\n{formatted_res}")
    elif packet.haslayer(TCP):
      tcp_payload = bytes(packet[TCP].payload)
      if b"HTTP" in tcp_payload or b"GET" in tcp_payload or b"POST" in tcp_payload:
        http_message = tcp_payload.decode('utf-8')
        print(http_message)
      else:
        print("Protocol is neither HTTP nor DNS")
    else:
      print("Protocol is neither HTTP nor DNS")

## Load packet capture (file with network traffic)

We use files containing network traffic made freely available by [Chris Sanders](https://github.com/chrissanders/packets).

To download the necessary files in this Linux virtual machine execute the following cell.

The `http_espn.pcapng` packet capture contains all the DNS and HTTP traffic generated by a browser for visualizing a certain web page. As we will see later, visualizing a web page requires fecthing several different files from several different web servers (one HTTP request-response pair for each file).

If you wanted to analyze other files that contain network traffic, you can upload them in the virtual machine as any other file (left section, click on the folder symbol, click on the upload symbol). Then, modify the argument of `rdpcap()` so that it operates on the corresponding file.


In [ ]:
!curl -O https://raw.githubusercontent.com/chrissanders/packets/master/http_espn.pcapng
packets = rdpcap('http_espn.pcapng')

# Navigating the web: localizing web servers

We do not know yet how the web works. For the moment, it suffices to know that the *browser* sends requests structured with a protocol called *HTTP* to *web servers*. The browser sends a HTTP request to a web server and waits for the matching HTTP response from that web server. The HTTP request specifies the document that the browser wants to download and the HTTP response contains that document.

Execute the next cell to print a summary of the initial part of the capture.

In [ ]:
!tshark -r http_espn.pcapng -Y "(dns or http)" -c 7

You can see that:

* Frame 6: the browser (172.16.0.122) sends an HTTP request to a web server (199.181.132.250)
* Frame 7: the web server sends an HTTP response to the browser

The exact meaning of HTTP requests and responses is irrelevant for the moment. The problem is: *how did the browser obtain the IP address of the web server*? This is a crucial question.

The answer is in the DNS traffic:

* Frame 1: the browser (172.16.0.122) sends a DNS request to its name server (4.2.2.1), asking to translate the name `www.espn.com`.
* Frame 2: the name server sends a DNS response with the corresponding IP address, i.e., 199.181.132.250

A question for you: *how did the browser obtain the IP address of the name server*?

**Important**: by removing option -c 7 in the tshark command above (or by using a value greater than 7) you would see a larger portion of the web traffic in the capture. This traffic is quite intricate and it may be confusing at this stage, though.

The reason is because the browser has to contact many different web servers and it does so in an order that is hard to understand (e.g., it sends a request to WS1, then to WS2, then to WS3, then it receives the response from WS2 and so on).

So, either do not look at the full capture or do not worry if you find it confusing.

The next cell lists *all* the DNS requests in the capture.



In [ ]:
!tshark -r http_espn.pcapng -Y "dns.flags.response == 0"

 Note that *all the DNS requests are sent to the same IP address*---i.e., the IP address of its default name server.

Please keep this in mind: an endpoint, such as a browser, always sends its DNS requests to its default name server: an endpoint never interacts with any other name server and, in particular, it never navigates on the domain tree starting from the name server of the root zone.

Also note that the set of queried names is identical to the set of web servers contacted by the browser (as shown in the previous cell).

# DNS traffic

The next cell shows the 3 first request-response pairs.

In [ ]:
!tshark -r http_espn.pcapng -Y "dns" -c 40

Note that each DNS response carries the requested RR, which means that the default name server has indeed obtained the requested RR on behalf of the browser (how the default name server has obtained this information is beyond the scope of this course).

Also note that the response to the third request is more complex than the other responses. This response states that the requested name `a.espncdn.com` is actually an alias of another name (`a.espncdn.com.edgesuite.net`), which is also an alias of yet another name (`a1831.g.akamai.net`) that is associated with two IP addresses.

The next cell displays this interaction in more detail (please ignore the `xid` field).

In [ ]:
indexes_to_display = [21,22]
display_frames(packets, indexes_to_display)

If we now print request-response pairs 4 and 5, we observe a strange behavior (execute the next cell).

In [ ]:
!tshark -r http_espn.pcapng -Y "dns && frame.number >= 200 && frame.number <= 230"

It can be seen that the browser sends a DNS request (frame 224) and then it sends *another* DNS request (225) *before* having received the response for the first one! The default name server then sends the response to the first request (226) and then it sends the response to the second request (227). In other words, the sequence of DNS traffic is REQ1, REQ2, RESP1, RESP2.

It is simple to imagine that things can become more complicated, for two key reasons:

1. In this case the browser has sent 2 requests before having received any response; in other cases the browser could send 3, 4 or more requests, before having received any response.

2. Responses need *not* be in the same order as the requests. For instance, there may be this sequence: REQ1, REQ2, RESP2, RESP1. Or, REQ1, REQ2, REQ3, RESP2, RESP3, RESP1. Or an even more complex sequence: REQ1, REQ2, RESP2, REQ3, REQ4, RESP3, RESP1, RESP4.

It is easy to realize that things can become very complicated: the flow of DNS requests and the flow of DNS responses can take many different forms.

In the exams we will assume that DNS traffic always follows the simplest pattern: one request-one response (REQ1,RESP1,REQ2,RESP2,REQ3,RESP3 and so on).

This notebook might convince you that although the course may seem difficult, reality is even more difficult.



## Out of order DNS traffic (optional)

Just for completeness and for those who might be interested.

Let us consider again the REQ1 REQ2 RESP1 RESP2 pattern observed above:

In [ ]:
!tshark -r http_espn.pcapng -Y "dns && frame.number >= 200 && frame.number <= 230"

The next cell displays the content of those frames in more detail.

In [ ]:
indexes_to_display = [224,225,226,227]
display_frames(packets, indexes_to_display)

It can be see that each DNS message carries a field called  `xid` that allows matching DNS responses to DNS requests, as follows.

The DNS client selects a unique identifier for each DNS request and inserts that identifier (`xid`) in the request. A DNS response contains the `xid` of the corresponding request.

By looking at the traffic above you can understand how this information allows the client to match responses with requests.

Executing the next cell will provide a listing of the full DNS traffic in the capture. You will see that there are more cases of "out of order" DNS traffic (try to identify them).


In [ ]:
!tshark -r http_espn.pcapng -Y "dns"

If you want, you can display the content of some out-of-order requests and response to see that they indeed contain the same `xid`

In [ ]:
indexes_to_display = [224,225,226,227]
display_frames(packets, indexes_to_display)